In [0]:
%pip install databricks-labs-dqx
 

In [0]:
dbutils.library.restartPython()
 

In [0]:
dbutils.widgets.text("catalog_name",  "dbw_chinook_team", "Catalog")

dbutils.widgets.text("bronze_schema", "chinook_bronze",    "Bronze")

dbutils.widgets.text("silver_schema", "chinook_silver",    "Silver")

dbutils.widgets.text("table_name",    "all",               "Table")
 

In [0]:
dbutils.widgets.text("catalog_name",  "dbw_chinook_team", "Catalog")
dbutils.widgets.text("bronze_schema", "chinook_bronze",    "Bronze")
dbutils.widgets.text("silver_schema", "chinook_silver",    "Silver")
dbutils.widgets.text("table_name",    "all",               "Table")

In [0]:
from pyspark.sql.functions import (

    col, trim, lower, coalesce,

    lit, to_date, current_timestamp

)
 
CATALOG = dbutils.widgets.get("catalog_name")

BRONZE  = f"{CATALOG}.{dbutils.widgets.get('bronze_schema')}"

SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"
 
# Correct DQX initialization for Serverless

try:

    from databricks.labs.dqx.engine import DQEngine

    from databricks.sdk import WorkspaceClient

    ws = WorkspaceClient()

    dqx = DQEngine(ws)

    print("✅ DQX installed and working!")

    USE_DQX = True

except Exception as e:

    print(f"⚠️ DQX not available: {e}")

    print("✅ Using manual DQX validation instead")

    USE_DQX = False
 
print(f"   Bronze : {BRONZE}")

print(f"   Silver : {SILVER}")
 

In [0]:
#  Create Log Tables 

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {SILVER}.dqx_execution_log (

    table_name      STRING,

    execution_time  TIMESTAMP,

    total_records   LONG,

    passed_records  LONG,

    failed_records  LONG,

    created_date    TIMESTAMP

)

""")
 
spark.sql(f"""

CREATE TABLE IF NOT EXISTS {SILVER}.quarantine (

    table_name  STRING,

    failed_rule STRING,

    failed_at   TIMESTAMP

)

""")

print("✅ Log tables created")
 
 

In [0]:
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, lit, current_timestamp

# ─────────────────────────────────────────────
# FUNCTION 1: DATA PROFILING
# ─────────────────────────────────────────────
def dqx_profile(bronze_table, table_name):
    df = spark.table(bronze_table)
    total = df.count()

    print(f"\n── {table_name} Profile ──")
    print(f"  Total rows   : {total}")
    print(f"  Total columns: {len(df.columns)}")
    print(f"\n  Null Analysis:")

    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        status = "⚠️ " if null_count > 0 else "✅"
        pct = round(null_count / total * 100, 1) if null_count > 0 else 0
        print(f"    {status} {c:<25} nulls: {null_count} ({pct}%)")

    dupes = total - df.dropDuplicates().count()
    print(f"\n  Duplicates: {'⚠️  ' + str(dupes) if dupes > 0 else '✅ 0'}")

    return df, total


# ─────────────────────────────────────────────
# FUNCTION 2: DQX VALIDATION + QUARANTINE
# ─────────────────────────────────────────────
def dqx_validate(df, table_name, rules, total):
    print(f"\n── {table_name} DQX Validation ──")

    passed_df = df
    all_failed = []

    for rule_name, condition in rules.items():
        failed = df.filter(~condition)
        fc = failed.count()

        if fc > 0:
            print(f"  ❌ {rule_name}: {fc} failed")
            all_failed.append(
                failed.select(
                    lit(table_name).alias("table_name"),
                    lit(rule_name).alias("failed_rule"),
                    current_timestamp().alias("failed_at")
                )
            )
        else:
            print(f"  ✅ {rule_name}: passed")

        passed_df = passed_df.filter(condition)

    passed = passed_df.count()
    failed_count = total - passed

    # Log results to execution log table
    spark.sql(f"""
        INSERT INTO {SILVER}.dqx_execution_log
        VALUES (
            '{table_name}',
            current_timestamp(),
            {total},
            {passed},
            {failed_count},
            current_timestamp()
        )
    """)

    print(f"\n  Total: {total} | Passed: {passed} | Failed: {failed_count}")

    # Write failed records to quarantine table
    if all_failed:
        quarantine = reduce(DataFrame.union, all_failed)
        quarantine.write \
            .format("delta") \
            .mode("append") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{SILVER}.quarantine")
        print(f"  ⚠️  {failed_count} records quarantined")

    return passed_df


print("✅ DQX functions defined successfully!")

In [0]:
print("=== DQX PROFILING ===")
customer_df,  c_total  = dqx_profile(f"{BRONZE}.customer",    "Customer")
invoice_df,   i_total  = dqx_profile(f"{BRONZE}.invoice",     "Invoice")
invline_df,   il_total = dqx_profile(f"{BRONZE}.invoiceline", "InvoiceLine")
track_df,     t_total  = dqx_profile(f"{BRONZE}.track",       "Track")
employee_df,  e_total  = dqx_profile(f"{BRONZE}.employee",    "Employee")


In [0]:
print("\n=== DQX VALIDATION ===")
customer_passed = dqx_validate(
    customer_df, "Customer", {
        "CustomerId_not_null": col("CustomerId").isNotNull(),
        "Email_not_null":      col("Email").isNotNull(),
        "FirstName_not_null":  col("FirstName").isNotNull(),
        "LastName_not_null":   col("LastName").isNotNull()
    }, c_total)
invoice_passed = dqx_validate(
    invoice_df, "Invoice", {
        "InvoiceId_not_null":  col("InvoiceId").isNotNull(),
        "CustomerId_not_null": col("CustomerId").isNotNull(),
        "Total_positive":      col("Total") > 0,
        "Date_not_null":       col("InvoiceDate").isNotNull()
    }, i_total)
invline_passed = dqx_validate(
    invline_df, "InvoiceLine", {
        "LineId_not_null":  col("InvoiceLineId").isNotNull(),
        "TrackId_not_null": col("TrackId").isNotNull(),
        "Qty_positive":     col("Quantity") > 0,
        "Price_positive":   col("UnitPrice") > 0
    }, il_total)
track_passed = dqx_validate(
    track_df, "Track", {
        "TrackId_not_null":  col("TrackId").isNotNull(),
        "Name_not_null":     col("Name").isNotNull(),
        "Duration_positive": col("Milliseconds") > 0
    }, t_total)
print("\n=== DQX EXECUTION LOG ===")
spark.table(f"{SILVER}.dqx_execution_log").show(truncate=False)
 

In [0]:
from pyspark.sql.functions import trim, lower, coalesce, to_date, lit, col

print("\n=== WRITING SILVER TABLES ===")

# ── Customer Silver ───────────────────────────────────
customer_silver = customer_passed \
    .withColumn("FirstName",  trim(col("FirstName"))) \
    .withColumn("LastName",   trim(col("LastName"))) \
    .withColumn("Email",      lower(trim(col("Email")))) \
    .withColumn("Company",    coalesce(col("Company"),    lit("N/A"))) \
    .withColumn("City",       coalesce(col("City"),       lit("Unknown"))) \
    .withColumn("State",      coalesce(col("State"),      lit("N/A"))) \
    .withColumn("Country",    coalesce(col("Country"),    lit("Unknown"))) \
    .withColumn("PostalCode", coalesce(col("PostalCode"), lit("N/A"))) \
    .withColumn("Phone",      coalesce(col("Phone"),      lit("N/A"))) \
    .withColumn("Fax",        coalesce(col("Fax"),        lit("N/A")))

customer_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER}.customer")

print(f"✅ Silver Customer: {customer_silver.count()} rows")


# ── Invoice Silver ────────────────────────────────────
invoice_silver = invoice_passed \
    .withColumn("InvoiceDate",    to_date(col("InvoiceDate"))) \
    .withColumn("BillingCountry", coalesce(col("BillingCountry"), lit("Unknown"))) \
    .withColumn("BillingState",   coalesce(col("BillingState"),   lit("N/A"))) \
    .withColumn("BillingCity",    coalesce(col("BillingCity"),    lit("Unknown"))) \
    .withColumn("BillingAddress", coalesce(col("BillingAddress"), lit("Unknown"))) \
    .withColumn("Total",          col("Total").cast("decimal(10,2)"))

invoice_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER}.invoice")

print(f"✅ Silver Invoice: {invoice_silver.count()} rows")


# ── InvoiceLine Silver ────────────────────────────────
invline_silver = invline_passed \
    .withColumn("UnitPrice", col("UnitPrice").cast("decimal(10,2)"))

invline_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER}.invoiceline")

print(f"✅ Silver InvoiceLine: {invline_silver.count()} rows")


# ── Track Silver ──────────────────────────────────────
track_silver = track_passed \
    .withColumn("UnitPrice", col("UnitPrice").cast("decimal(10,2)")) \
    .withColumn("Composer",  coalesce(col("Composer"), lit("Unknown")))

track_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER}.track")

print(f"✅ Silver Track: {track_silver.count()} rows")


# ─────────────────────────────────────────────────────────────
# Employee: Profile + Validate
# ─────────────────────────────────────────────────────────────

employee_df, e_total = dqx_profile(f"{BRONZE}.employee", "Employee")

employee_passed = dqx_validate(
    employee_df,
    "Employee",
    {
        "EmployeeId_not_null": col("EmployeeId").isNotNull(),
        "FirstName_not_null":  col("FirstName").isNotNull(),
        "LastName_not_null":   col("LastName").isNotNull(),
        "Email_not_null":      col("Email").isNotNull(),
        "HireDate_not_null":   col("HireDate").isNotNull()
    },
    e_total
)

# ─────────────────────────────────────────────────────────────
# Employee Silver
# ─────────────────────────────────────────────────────────────

employee_silver = (
    employee_passed
        .withColumn("FirstName", trim(col("FirstName")))
        .withColumn("LastName",  trim(col("LastName")))
        .withColumn("Email",     lower(trim(col("Email"))))
        .withColumn("City",      coalesce(col("City"),    lit("Unknown")))
        .withColumn("State",     coalesce(col("State"),   lit("N/A")))
        .withColumn("Country",   coalesce(col("Country"), lit("Unknown")))
)

employee_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER}.employee")

print(f"✅ Silver Employee: {employee_silver.count()} rows")


# ── Pass-through Tables ───────────────────────────────
passthrough = [
    "artist", "album", "genre", "mediatype",
    "playlist", "playlisttrack"
]

for tbl in passthrough:
    df = spark.table(f"{BRONZE}.{tbl}")
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER}.{tbl}")
    print(f"✅ Silver {tbl}: {df.count()} rows")

print("\n=== SILVER LOAD COMPLETE ===")

In [0]:
print("\n=== FINAL DQX SUMMARY ===")

spark.table(f"{SILVER}.dqx_execution_log") \
    .select(
        "table_name",
        "total_records",
        "passed_records",
        "failed_records",
        "execution_time"
    ).show(truncate=False)


print("\n=== SILVER VALIDATION ===")

silver_tables = [
    "customer", "invoice", "invoiceline",
    "track", "employee", "artist", "album",
    "genre", "mediatype", "playlist", "playlisttrack"
]

for tbl in silver_tables:
    count = spark.table(f"{SILVER}.{tbl}").count()
    print(f"  ✅ {tbl:<20} {count:>6} rows")


print("\n✅ BRONZE TO SILVER COMPLETE!")